# Работа с API

In [1]:
import requests
import pandas as pd

In [2]:
API_KEY = 'XK01BFB-ZMSMN4R-PJV5WVF-T1DWT89'
BASE_URL = 'https://api.poiskkino.dev/v1/movie'
HEADERS = {'X-API-KEY': API_KEY}

In [3]:
result_list = {'title': [], 'genre': [], 'year': [],
               'country': [], 'rate': [], 'description': []
              }
params = {'limit': 250,
          'selectFields': ['name', 'genres', 'year', 'countries', 'rating', 'description']
         }

try:
    response = requests.get(BASE_URL, headers=HEADERS, params=params)
    response.raise_for_status()
    data = response.json()

    for movie in data.get('docs', []):
        result_list['title'].append(movie.get('name', '—'))
        result_list['year'].append(movie.get('year', '—'))
        result_list['description'].append(movie.get('description', '—'))
        
        genres = [g.get('name') for g in movie.get('genres', [])]
        result_list['genre'].append(", ".join(genres) if genres else "—")
        
        countries = [c.get('name') for c in movie.get('countries', [])]
        result_list['country'].append(", ".join(countries) if countries else "—")
        
        rate = movie.get('rating', {}).get('kp', 0.0)
        result_list['rate'].append(rate)

    df = pd.DataFrame(result_list)

except Exception as e:
    print(f"Ошибка запроса: {e}")

In [4]:
for key, value in result_list.items():
    print(f"{key}: {len(value)}")

title: 250
genre: 250
year: 250
country: 250
rate: 250
description: 250


In [5]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
genre - 0
year - 0
country - 0
rate - 0
description - 0


In [6]:
df.to_csv('Top250_API_Data.csv', index=False)
df.head(10)

,title,genre,year,country,rate,description
0,1+1,"драма, комедия",2011,Франция,8.859,"Пострадав в результате несчастного случая, бог..."
1,Джентльмены,"криминал, комедия, боевик",2019,"США, Великобритания",8.673,Один ушлый американец ещё со студенческих лет ...
2,Триггер,драма,2018,Россия,8.450,Психолог Артём Стрелецкий — сторонник шоковой ...
3,Гнев человеческий,"боевик, триллер",2021,"Великобритания, США",7.710,Грузовики лос-анджелесской инкассаторской комп...
4,Брат,"драма, криминал, боевик",1997,Россия,8.370,"Демобилизовавшись, Данила Багров возвращается ..."
5,Волк с Уолл-стрит,"драма, криминал, биография, комедия",2013,США,8.095,1987 год. Джордан Белфорт становится брокером ...
6,Фишер,"детектив, драма, криминал, триллер",2023,Россия,7.854,"Интеллигентный следователь Валерий Козырев, ег..."
7,Игра престолов,"фэнтези, драма, боевик, мелодрама, приключения",2011,"США, Великобритания",9.011,"К концу подходит время благоденствия, и лето, ..."
8,Достать ножи,"детектив, комедия, драма, криминал",2019,США,8.188,На следующее утро после празднования 85-летия ...
9,Зеленая книга,"биография, комедия, драма",2018,"США, Китай",8.547,1960-е годы. После закрытия нью-йоркского ночн...


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        250 non-null    object 
 1   genre        250 non-null    object 
 2   year         250 non-null    int64  
 3   country      250 non-null    object 
 4   rate         250 non-null    float64
 5   description  250 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 11.8+ KB


# Объединение файлов

In [12]:
file_html = 'Top250FilmsInPremier.csv'
file_api = 'Top250_API_Data.csv'

df1 = pd.read_csv(file_html)
df2 = pd.read_csv(file_api)

df_combined = pd.concat([df2, df1], ignore_index=True)

df_combined.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)

df_combined['rate'] = pd.to_numeric(df_combined['rate'], errors='coerce').fillna(0.0)
df_combined['rate'] = df_combined['rate'].round(1)

df_combined.drop_duplicates(subset=['title', 'year'], keep='first', inplace=True)
df_combined.dropna(subset=['title'], inplace=True)

df_combined.fillna({'genre': '—', 'country': '—', 'description': 'Нет описания'}, inplace=True)

output_file = 'Final_Movies_Dataset.csv'
df_combined.to_csv(output_file, index=False)

print(f"Итого записей в финальном файле: {len(df_combined)}")

Итого записей в финальном файле: 489


In [13]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   250 non-null    int64  
 1   title        250 non-null    object 
 2   genre        250 non-null    object 
 3   year         250 non-null    int64  
 4   country      250 non-null    object 
 5   rate         250 non-null    float64
 6   description  250 non-null    object 
dtypes: float64(1), int64(2), object(4)
memory usage: 13.8+ KB


In [14]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        250 non-null    object 
 1   genre        250 non-null    object 
 2   year         250 non-null    int64  
 3   country      250 non-null    object 
 4   rate         250 non-null    float64
 5   description  250 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 11.8+ KB


In [15]:
df3 = pd.read_csv('Final_Movies_Dataset.csv')
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 489 entries, 0 to 488
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        489 non-null    object 
 1   genre        489 non-null    object 
 2   year         489 non-null    int64  
 3   country      489 non-null    object 
 4   rate         489 non-null    float64
 5   description  489 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 23.1+ KB
